In [1]:
"""
06_exploratory_analysis.py

Compare healthy vs. failed observations on the financial ratios:
mean/median by group, correlations, Mann-Whitney U tests, and a spot
check on the "dgrmovestreet" behavioural variable.
"""


'\n06_exploratory_analysis.py\n\nCompare healthy vs. failed observations on the financial ratios:\nmean/median by group, correlations, Mann-Whitney U tests, and a spot\ncheck on the "dgrmovestreet" behavioural variable.\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd
from scipy.stats import mannwhitneyu

from config import FEATURES_FINANCIAL, TARGET
from utils.print_section import print_section

print_section("Mean")
print(df.groupby(TARGET)[FEATURES_FINANCIAL].mean())

print_section("Median")
median_features = [f for f in FEATURES_FINANCIAL if f not in ("log_age", "size")]
print(df.groupby(TARGET)[median_features].median())

print_section("Correlations")
print(df[FEATURES_FINANCIAL].corr())

# ------------------------------------------------------------------
# Mann-Whitney U tests: is each feature's distribution significantly
# different between healthy (target=0) and failed (target=1) firms?
# ------------------------------------------------------------------
print_section("Mann-Whitney U tests")

results = []
for feature in FEATURES_FINANCIAL:
    healthy = df.loc[df[TARGET] == 0, feature].dropna()
    failed = df.loc[df[TARGET] == 1, feature].dropna()

    _, p_value = mannwhitneyu(healthy, failed, alternative="two-sided")

    results.append({
        "feature": feature,
        "p_value": p_value,
        "healthy_median": healthy.median(),
        "failed_median": failed.median(),
    })

print(pd.DataFrame(results))



Mean
        profitability  liquidity   solvency  structure   log_age       size
target                                                                     
0           -0.551703  20.432005 -40.185019   0.739950  2.443892  12.212406
1           -5.118872   5.505858 -64.104332   0.855212  2.323365  11.196500

Median
        profitability  liquidity  solvency  structure
target                                               
0            0.036218   1.500386  0.381471   0.892597
1           -0.154000   0.655377 -0.198374   0.991753

Correlations
               profitability  liquidity  solvency  structure   log_age  \
profitability       1.000000   0.000024 -0.373555  -0.002697  0.000853   
liquidity           0.000024   1.000000  0.000033  -0.000995  0.002359   
solvency           -0.373555   0.000033  1.000000   0.003654 -0.001844   
structure          -0.002697  -0.000995  0.003654   1.000000 -0.009260   
log_age             0.000853   0.002359 -0.001844  -0.009260  1.000000   
size    

         feature        p_value  healthy_median  failed_median
0  profitability   0.000000e+00        0.036218      -0.154000
1      liquidity   0.000000e+00        1.500386       0.655377
2       solvency   0.000000e+00        0.381471      -0.198374
3      structure   2.635645e-83        0.892597       0.991753
4        log_age   2.913947e-22        2.484907       2.302585
5           size  1.254591e-206       12.287828      11.343974
